In [2]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import wandb

# ==========================================================
# 1. ENVIRONMENT & PATH RESOLUTION
# ==========================================================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
REPO_PATH = os.path.join(BASE_DIR, 'src', 'models', 'Depth-Anything-V2')

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

from depth_anything_v2.dpt import DepthAnythingV2

# Define target data directory splits managed by DVC
DATA_DIR = os.path.join(BASE_DIR, "src", "data")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train", "images")
TRAIN_DEPTH_DIR = os.path.join(DATA_DIR, "train", "depths")
VAL_IMG_DIR = os.path.join(DATA_DIR, "validate", "images")
VAL_DEPTH_DIR = os.path.join(DATA_DIR, "validate", "depths")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==========================================================
# 2. DETE_DEPENDENT LOSS FUNCTION: SILog Loss
# ==========================================================
class SILogLoss(nn.Module):
    """
    Scale-Invariant Logarithmic Loss (SILog)
    Penalizes relative variance errors, focusing accuracy heavily on foreground objects.
    """
    def __init__(self, lambd=0.5, eps=1e-6):
        super().__init__()
        self.lambd = lambd
        self.eps = eps

    def forward(self, pred, target):
        # Valid mask filters out missing depth readings
        valid_mask = (target > self.eps) & (pred > self.eps)
        if not valid_mask.any():
            return torch.tensor(0.0, requires_grad=True, device=pred.device)

        log_pred = torch.log(pred[valid_mask])
        log_target = torch.log(target[valid_mask])
        
        diff = log_pred - log_target
        
        variance_term = torch.mean(diff ** 2)
        scale_consistency_term = (self.lambd / (torch.numel(diff) ** 2)) * (torch.sum(diff) ** 2)
        
        return variance_term - scale_consistency_term

# ==========================================================
# 3. PYTORCH DATA INGESTION ENGINE
# ==========================================================
class NYUDepthDataset(Dataset):
    def __init__(self, img_dir, depth_dir, input_size=518):
        self.img_dir = img_dir
        self.depth_dir = depth_dir
        self.input_size = input_size
        self.filenames = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        # Load and normalize raw RGB images
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.resize(img, (self.input_size, self.input_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        
        # ImageNet normalization standard
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()

        # Load target ground truth metric depth matrix maps
        depth_path = os.path.join(self.depth_dir, fname.split('.')[0] + '.png')
        if not os.path.exists(depth_path):
            depth_path = os.path.join(self.depth_dir, fname.split('.')[0] + '.npy')
            
        if depth_path.endswith('.npy'):
            depth = np.load(depth_path)
        else:
            depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
            
        depth = cv2.resize(depth, (self.input_size, self.input_size), interpolation=cv2.INTER_NEAREST)
        depth_tensor = torch.from_numpy(depth).float().unsqueeze(0)

        return img_tensor, depth_tensor

# ==========================================================
# 4. HYBRID MODEL FRAMEWORK CREATION
# ==========================================================
class CustomHybridModel(nn.Module):
    def __init__(self, encoder='vitb'):
        super().__init__()
        config = {'encoder': encoder, 'features': 128, 'out_channels': [96, 192, 384, 768]}
        self.backbone = DepthAnythingV2(**config)
        
        # Trainable CNN Decoder Mapping Layer
        self.depth_head = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )
        
    def forward(self, x):
        # Extract features through frozen base transformer
        with torch.no_grad():
            raw_depth = self.backbone(x)
            
        if raw_depth.ndim == 3:
            raw_depth = raw_depth.unsqueeze(1)
            
        return self.depth_head(raw_depth)

# ==========================================================
# 5. CORE EXECUTION MANAGEMENT PIPELINE
# ==========================================================
def run_retrain_pipeline():
    # --- Hyperparameters Config ---
    EPOCHS = 30
    BATCH_SIZE = 8
    LEARNING_RATE = 1e-4

    print(f"🚀 Initializing Weights & Biases Tracking System...")
    wandb.init(
        project="Monocular-3D-Reconstruction",
        config={
            "architecture": "DA-V2-ViT-B + Custom CNN Head",
            "loss_function": "SILogLoss",
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE
        }
    )

    print("📦 Loading Datasets...")
    train_dataset = NYUDepthDataset(TRAIN_IMG_DIR, TRAIN_DEPTH_DIR)
    val_dataset = NYUDepthDataset(VAL_IMG_DIR, VAL_DEPTH_DIR)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"🔧 Building Hybrid Model Shell on Device: {DEVICE}...")
    model = CustomHybridModel(encoder='vitb')
    
    # Freeze the transformer core completely
    for param in model.backbone.parameters():
        param.requires_grad = False
        
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.depth_head.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
    criterion = SILogLoss()

    best_val_loss = float('inf')
    checkpoints_output_dir = os.path.join(BASE_DIR, "checkpoints")
    os.makedirs(checkpoints_output_dir, exist_ok=True)
    save_path = os.path.join(checkpoints_output_dir, "latest_hybrid_model.pth")

    print("\n🏋️ Starting Training Loop Operations...")
    for epoch in range(EPOCHS):
        model.train()
        running_train_loss = 0.0
        
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            predictions = model(imgs)
            loss = criterion(predictions, targets)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()

        # --- Validation Evaluation Pass ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for val_imgs, val_targets in val_loader:
                val_imgs, val_targets = val_imgs.to(DEVICE), val_targets.to(DEVICE)
                val_preds = model(val_imgs)
                val_loss = criterion(val_preds, val_targets)
                running_val_loss += val_loss.item()

        epoch_train_loss = running_train_loss / len(train_loader)
        epoch_val_loss = running_val_loss / len(val_loader)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
        
        # Log active metric tracking telemetry instantly to W&B
        wandb.log({"train_loss": epoch_train_loss, "val_loss": epoch_val_loss, "epoch": epoch + 1})

        # Checkpointing Strategy: Save only the best generalizing model state
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), save_path)
            print(f"💾 Guarded Checkpoint Updated! Saved Best Weights to: {save_path}")

    print("\n📦 Registering Final Trained Weights to W&B Artifact Server Vault...")
    artifact = wandb.Artifact(name="depth_anything_hybrid_ensemble", type="model", description="Fully trained CNN head model.")
    artifact.add_file(save_path)
    wandb.log_artifact(artifact)

    wandb.finish()
    print("🎉 Retraining Sequence Successfully Completed!")

if __name__ == "__main__":
    run_retrain_pipeline()

ModuleNotFoundError: No module named 'depth_anything_v2'